

In this notebook: 
- What is a Spark DataFrame? 
- Creating our first DataFrame
- Schemas
- Reading and writing

## What is a Spark DataFrame? 

A Spark DataFrame is like a distributed, in-memory table with **named columns** and a **schema**. 

The schema defines the columns and the data types for each column. 

Inspired by Pandas DataFrames! 

## Creating our first data frame (with a schema)


In [ ]:
from pyspark.sql.types import StructType,StructField, StringType, IntegerType

data2 = [("Jack","","Eldridge","36636","M",90000),
    ("Matthew","J", "Munro","28832","M",45400),
    ("Sheila","Oway", "Roberts","12114","F",64000),
    ("Anne","", "Dushane","32192","F",141000),
    ("Jane","Rebecca","Jones","99482","F",56000)
  ]

schema = StructType([ 
    StructField("firstname",StringType(),True), 
    StructField("middlename",StringType(),True), 
    StructField("lastname",StringType(),True), 
    StructField("id", StringType(), True), 
    StructField("gender", StringType(), True), 
    StructField("salary", IntegerType(), True) 
  ])
 
df = spark.createDataFrame(data=data2,schema=schema)
df.printSchema()
df.show(truncate=False)


In [ ]:
type(df)

## Why Schemas? 
- Commonly used, especially with reading from an external data source (including files). 
- Spark doesn't have to 'infer' the data type (which can be expensive). 
- You can detect errors early if the data doesn't match the schema.  

#### Defining a schema using Data Definition Language (DDL) 

In [ ]:
schema_ddl = "firstname STRING, middlename STRING, lastname STRING, id STRING, gender STRING, salary INT " 
df_with_ddl_schema = spark.createDataFrame(data=data2,schema=schema_ddl)
df.printSchema()


## Reading a CSV file into a Spark DataFrame
- Basic read
- with headers, 
- inferring the schema 


Using property-sales.csv, which looks like this: 

Address,Type,City,SalePrice ($),Agent

1 Rowley Street ,Detached House,New York,745000,Penelope Pullman 

13a lollipop avenue,Apartment,Los Angeles,345000,Jack Smith 

34 the drive,House,Atlanta,459000,Sheila Sammi

In [ ]:
# Declare the path to our file 
csv_path = 'Files/property-sales.csv' 

# Read a csv file from Files/property-sales.csv
df_csv = spark.read.csv(csv_path, header=True) 

display(df_csv)

In [ ]:
df_csv.write.mode("overwrite").format("csv").save("Files/ " + csv_table_name)


In [ ]:
df_csv.dtypes

## Writing DataFrames to files (JSON) 
We can write out our DataFrame as a JSON file by calling df.write.json() 

In [ ]:
# call write.json() on our 
df_csv.write.json("Files/json/property-sales.json", mode='overwrite')


## Reading from JSON File into DataFrame

In [ ]:
df_json = spark.read.json('Files/json/property-sales.json')
display(df_json)

## Writing out to parquet

In [ ]:
df_json.write.parquet('Files/parquet/property-sales2.parquet', mode='overwrite')

Reading in parquet into a DataFrame

In [ ]:
df_parquet = spark.read.parquet('Files/parquet/property-sales.parquet')
display(df_parquet)

## Reading multiple files in the same folder
- creating multiple parquet files in the parquet subfolder first
- read in all the parquet files into one df 

In [ ]:
# read all the parquet files in the 'Files/parquet/' folder into a dataframe  
df_all_parquet = spark.read.parquet('Files/parquet/*.parquet')

## Checking this has worked using _metadata
Spark provides us with all the file metadata in a 'hidden' column that we can add to our dataframe using _metadata. 

In [ ]:
# read all the parquet files, then add the _metadata column 
df_all_parquet_plus_metadata = spark.read\
    .parquet('Files/parquet/*.parquet')\
    .select("*", "_metadata")

display(df_all_parquet_plus_metadata)